# Digital Image Processing Project
## Fruit Sorting System by Color and Size

**Objective**: Create an industrial pipeline to sort fruits using advanced image processing techniques.

---

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from typing import Optional, Tuple
from google.colab import drive

# Constants for tuning
GAUSSIAN_KERNEL = (5, 5)
MEDIAN_KSIZE = 5
FIGURE_SIZE = (12, 6)

## Section 1: Pre-processing
การปรับปรุงคุณภาพภาพเพื่อลด Noise และเพิ่ม Contrast ให้วัตถุแยกออกจากพื้นหลังได้ชัดเจนที่สุด

### 1.1 Helper Functions (Visualization)
สร้างฟังก์ชันสำหรับการแสดงผลภาพเปรียบเทียบแบบ Side-by-Side

In [ ]:
def show_comparison(img1: np.ndarray, img2: np.ndarray, title1: str = "Original", title2: str = "Processed") -> None:
    """Displays two images side by side for easy comparison."""
    plt.figure(figsize=FIGURE_SIZE)
    
    # Left image
    plt.subplot(1, 2, 1)
    if len(img1.shape) == 3:
        plt.imshow(cv2.cvtColor(img1, cv2.COLOR_BGR2RGB))
    else:
        plt.imshow(img1, cmap='gray')
    plt.title(title1)
    plt.axis('off')
    
    # Right image
    plt.subplot(1, 2, 2)
    if len(img2.shape) == 3:
        plt.imshow(cv2.cvtColor(img2, cv2.COLOR_BGR2RGB))
    else:
        plt.imshow(img2, cmap='gray')
    plt.title(title2)
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()

### 1.2 Image Acquisition

In [ ]:
def load_image(path: str) -> Optional[np.ndarray]:
    """Loads an image with basic error handling."""
    try:
        img = cv2.imread(path)
        if img is None:
            raise FileNotFoundError(f"Cannot find image at {path}")
        return img
    except Exception as e:
        print(f"Error: {e}")
        return None

### 1.3 Histogram Equalization (Manual Implementation)
การใช้ CDF เพื่อกระจายความเข้มแสงให้ทั่วภาพ ช่วยให้รายละเอียดในส่วนที่มืดหรือสว่างเกินไปชัดเจนขึ้น

In [ ]:
def manual_histogram_equalization(gray_img: np.ndarray) -> np.ndarray:
    """Manually implements global histogram equalization."""
    # Ensure image is grayscale
    if len(gray_img.shape) != 2:
        gray_img = cv2.cvtColor(gray_img, cv2.COLOR_BGR2GRAY)
        
    # 1. Histogram Calculation
    hist, _ = np.histogram(gray_img.flatten(), 256, [0, 256])
    
    # 2. CDF Calculation
    cdf = hist.cumsum()
    
    # 3. CDF Normalization (Formula based on DIP Theory)
    cdf_m = np.ma.masked_equal(cdf, 0)
    cdf_m = (cdf_m - cdf_m.min()) * 255 / (cdf_m.max() - cdf_m.min())
    equalized_lookup_table = np.ma.filled(cdf_m, 0).astype('uint8')
    
    # 4. Image Mapping
    return equalized_lookup_table[gray_img]

### 1.4 Noise Reduction Filters

In [ ]:
def denoise_image(img: np.ndarray, method: str = 'gaussian') -> np.ndarray:
    """
    Reduces noise using specified method.
    'gaussian': Smooths general electronic noise.
    'median': Best for salt-and-pepper noise.
    """
    if method.lower() == 'gaussian':
        return cv2.GaussianBlur(img, GAUSSIAN_KERNEL, 0)
    elif method.lower() == 'median':
        return cv2.medianBlur(img, MEDIAN_KSIZE)
    else:
        return img